# Land Cover Mapping Methods Comparison

## Pre-requisistes
- Earth Engine Initialization
- AOI retrieval

In [1]:
import ee
import luma_ge
ee.Authenticate() #force=True use for re-authentication
ee.Initialize()

In [2]:
#############################  Area of Interest  ###########################
def get_aoi_from_gaul(country="Indonesia", province="Sumatera Selatan"):
    """
    Get Area of Interest geometry from GAUL administrative boundaries.
    
    Parameters:
    -----------
    country : str
        Country name (default: "Indonesia")
    province : str
        Province/state name (default: "Sumatera Selatan")
        
    Returns:
    --------
    ee.Geometry : Area of interest geometry
    """
    admin = ee.FeatureCollection("FAO/GAUL/2015/level1")
    aoi_fc = admin.filter(ee.Filter.eq('ADM0_NAME', country)).filter(
        ee.Filter.eq('ADM1_NAME', province)
    )
    return aoi_fc.geometry()
aoi = get_aoi_from_gaul()


## Direct Classification
1. Search The Imagery 
2. Define the predictor
3. Run the classification

### Landsat Classification

In [4]:
#import the library
import geemap
from luma_ge.data_acquisition import Reflectance_Data, final_Image
aoi = geemap.shp_to_ee('../data/ref_data_testing/AOI_Oganilir.shp')
#initilize reflectance data retrieval class
reflectance = Reflectance_Data()
#retrieve the image collection
#use 2016-2017 image for better coverage
img_col, stat = reflectance.get_optical_data(aoi, #Aoi
                                       '2017-01-01', '2017-12-31', #start, end
                                       optical_data='L8_SR', #sensor
                                       cloud_cover=50, #cloud cover
                                       compute_detailed_stats=False)
#get the thermal band
thermal, stat = reflectance.get_thermal_bands(aoi, 
                                              '2017-01-01', '2017-12-31',
                                              thermal_data='L8_TOA',
                                              cloud_cover=50,
                                              compute_detailed_stats=False)
#initilize the compositing class
comp = final_Image()
#create the composite for multispectral and thermal data
med_landsat = comp.get_temporal_composite(img_col, aoi, reducer='Median')
median_landsat = med_landsat.select(
    med_landsat.bandNames().remove('AEROSOL')
)
thermal_median = comp.get_temporal_composite(thermal, aoi )
#define the visulization parameter and show them on the map
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}
map = geemap.Map()
map.centerObject(aoi, zoom=10)
map.addLayer(thermal_median, {}, "Thermal")
map.addLayer(img_col,l8_sr_visparam, "Collection")
map.addLayer(median_landsat, l8_sr_visparam, "Median Image")
map

2026-04-23 10:10:51,067 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-04-23 10:10:51,068 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-04-23 10:10:51,069 - Reflectance_Data - INFO - Date range: 2017-01-01 to 2017-12-31
2026-04-23 10:10:51,070 - Reflectance_Data - INFO - Cloud cover threshold: 50%
2026-04-23 10:10:51,070 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-04-23 10:10:51,070 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-04-23 10:10:51,072 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-04-23 10:10:51,074 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-04-23 10:10:51,075 - Reflectance_Data - INFO - Starting thermal data fetch for Landsat 8 Top-of-atmosphere reflectance
2026-04-23 10:10:51,075 - Reflectance_Data - INFO - Date range: 2017-01-01 to 2017-12-31
2026-04-23 1

Map(center=[-3.4152616959981223, 104.60534276364234], controls=(WidgetControl(options=['position', 'transparen…

#### Terrain and Spectral Predictors

In [4]:
from luma_ge.predictor import terrain_calculator, SpectralCalculator
# Initialize predictor backends
terrain_calc = terrain_calculator()
spectral_calc = SpectralCalculator()

# Choose DEM source and compute terrain layers
dem_source = 'NASADEM'
elevation = terrain_calc.calculate_elevation(aoi, dem_source=dem_source)
slope = terrain_calc.calculate_slope(aoi, dem_source=dem_source)
aspect = terrain_calc.calculate_aspect(aoi, dem_source=dem_source)

# Visualize terrain layers on the map
terrain_vis = {
    'min': 0,
    'max': 500,
    'palette': ['black', 'blue', 'green', 'yellow', 'red']
}
slope_vis = {
    'min': 0,
    'max': 45,
    'palette': ['white', 'lightblue', 'green', 'yellow', 'red']
}
aspect_vis = {
    'min': 0,
    'max': 360,
    'palette': ['white', 'blue', 'green', 'yellow', 'red', 'purple']
}

map.addLayer(elevation, terrain_vis, f"Elevation ({dem_source})")
map.addLayer(slope, slope_vis, f"Slope ({dem_source})")
#map.addLayer(aspect, aspect_vis, f"Aspect ({dem_source})")

# Compute spectral indices from the optical collection
indices_to_compute = ['EVI', 'GBNDVI', 'MSAVI', 'NDVI', 'MNDWI', 'AWEInsh', 'NDBI', 'NDMI', 'MBI' ]
spectral_indices = spectral_calc.calculate_indices_with_collection(
    collection=img_col,
    aoi=aoi,
    index_list=indices_to_compute,
    reducer_method='mean'
)

print('Computed terrain layers: elevation, slope, aspect')
print('Computed spectral indices:', spectral_indices.bandNames().getInfo() if spectral_indices else 'failed')

if spectral_indices:
    ndvi_vis = {'min': -1, 'max': 1, 'palette': ['purple', 'white', 'green']}
    evi_vis = {'min': -1, 'max': 1, 'palette': ['navy', 'white', 'lime']}
    map.addLayer(spectral_indices.select('GBNDVI'), ndvi_vis, 'GBNDVI')
    map.addLayer(spectral_indices.select('EVI'), evi_vis, 'EVI')
#stack the predictors and the imagery
predictor_stack = median_landsat.addBands(thermal_median).addBands(spectral_indices).addBands(elevation).addBands(slope).toFloat()
print('Final predictor stack bands:', predictor_stack.bandNames().getInfo() if predictor_stack else 'failed')


2026-04-22 13:39:12,842 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-04-22 13:39:14,011 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-04-22 13:39:14,013 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-04-22 13:39:14,013 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-04-22 13:39:14,014 - luma_ge.predictor - INFO - Calculating slope layer using NASADEM DEM...
2026-04-22 13:39:14,014 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-04-22 13:39:14,015 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-04-22 13:39:14,015 - luma_ge.predictor - INFO - Successfully calculated slope layer using NASADEM DEM
2026-04-22 13:39:14,015 - luma_ge.predictor - INFO - Calculating aspect layer using NASADEM DEM...
2026-04-22 13:39:14,016 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...


Computed terrain layers: elevation, slope, aspect
Computed spectral indices: ['EVI', 'GBNDVI', 'MSAVI', 'NDVI', 'MNDWI', 'AWEInsh', 'NDBI', 'NDMI', 'MBI']
Final predictor stack bands: ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'THERMAL', 'EVI', 'GBNDVI', 'MSAVI', 'NDVI', 'MNDWI', 'AWEInsh', 'NDBI', 'NDMI', 'MBI', 'elevation', 'slope']


#### Classification 

In [ ]:
#extract the pixel sample
from luma_ge.classification import FeatureExtraction, Generate_LULC
#samples
sample = geemap.shp_to_ee('../data/ref_data_testing/GT_2016_epistem_OganIlir.shp')
features = FeatureExtraction()
train, test = features.stratified_split(sample, predictor_stack, 
                            class_prop='ID_epistem', train_ratio=0.7, seed=42)

2026-04-22 13:40:01,511 - pyogrio._io - INFO - Created 700 records


Stratified Random Split Training Pixel Size: 503
Stratified Random Split Testing Pixel Size: 197


In [ ]:
#initilizae the classification class
clf = Generate_LULC()
#applied hard classification/original MS
classification_raw, model_raw = clf.hard_classification(training_data = train, 
 class_property='ID_epistem',
 image=predictor_stack, 
 ntrees=350,
 v_split=9,
 return_model=True)
#Land cover class definition
lc_class = {
    1: {"name": "Secondary Dryland Forest",   "color": "#054504"},
    6: {"name": "Secondary Swamp Forest",   "color": "#059486"},
    16: {"name": "Mixed/home Garden", "color": "#0deb50"},
    15: {"name": "Rubber Agroforest",    "color": "#839248"},
    #14: {"name": "Coffee Agroforest",    "color": "#df980a"},
    #7: {"name": "Plantation Forest",    "color": "#09a726"},
    9: {"name": "Oil Palm Monoculture",    "color": "#d9cc66"},
    8: {"name": "Rubber monoculture",    "color": "#414127"},
    11: {"name": "Coconut monoculture",    "color": "#d9e66c"},
    12: {"name": "Other monoculture",    "color": "#6aa66d"},
    17: {"name": "Paddy Field",    "color": "#b1eb03"},
    13: {"name": "Other Cropland",    "color": "#f1d900"},
    18: {"name": "Grass or Savanna",    "color": "#bdf2c0"},    
    21: {"name": "Cleared land",    "color": "#413d2f"},
    20: {"name": "Settlement",    "color": "#e00c0c"},
    #24: {"name": "Fish Pond",    "color": "#e18adb"},
    23: {"name": "Water body",    "color": "#0b3bdb"},                  
}
ogan_ilir_class = {
    1: {"name": "Secondary Dryland Forest",   "color": "#054504"},
    6: {"name": "Secondary Swamp Forest",   "color": "#059486"},
    8: {"name": "Rubber monoculture",    "color": "#583405"},
    9: {"name": "Oil Palm Monoculture",    "color": "#e0a911"},
    13: {"name": "Other Cropland",    "color": "#f1d900"},
    15: {"name": "Rubber Agroforest",    "color": "#839248"},
    16: {"name": "Mixed/home Garden", "color": "#0deb50"},
    17: {"name": "Paddy Field",    "color": "#b1eb03"},
    18: {"name": "Grass or Savanna",    "color": "#bdf2c0"},
    20: {"name": "Settlement",    "color": "#e00c0c"},
    21: {"name": "Cleared land",    "color": "#464440"},
    23: {"name": "Water body",    "color": "#0b3bdb"},
}

In [ ]:
#Evaluate model performance
print("Evaluating raw classification.")
try:
    model_acc_first = clf.evaluate_model(
        trained_model=model_raw,
        test_data=test,
        class_property='ID_epistem'
    )
    
except Exception as e:
    print(f"❌ Error in model evaluation: {e}")                                                    
import pandas as pd
# Display accuracy results
print("=== Model Performance Summary ===")
print(f"Overall Accuracy: {model_acc_first['overall_accuracy']:.4f} ({model_acc_first['overall_accuracy']*100:.2f}%)")
print(f"Kappa Coefficient: {model_acc_first['kappa']:.4f}")
print(f"Overall G-Mean: {model_acc_first['overall_gmean']:.4f}")
print("\n=== Per-Class Metrics ===")
#Class Dataframe
metrics_df = pd.DataFrame({
    'Precision': model_acc_first['precision'],
    'Recall': model_acc_first['recall'],
    'F1-Score': model_acc_first['f1_scores'],
    'G-Mean': model_acc_first['gmean_per_class']
})
#Round to 4 decimal places
metrics_df = metrics_df.round(4)
display(metrics_df)

Overall Accuracy: 0.4561 (45.61%)
Kappa Coefficient: 0.3600

In [ ]:
#visualize the feature importance
import plotly.express as px
importance = clf.get_feature_importance(model_raw, training_data=train, class_property='ID_epistem')
fig = px.bar(
                importance,
                x='Importance',
                y='Band',
                orientation='h',
                title='Kanal mana yang paling penting?',
                color='Importance',
                color_continuous_scale='Viridis',
                text='Importance'
            )
            
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(
                yaxis={'categoryorder': 'total ascending'},
                height=max(400, len(importance) * 30),
                showlegend=False
            )
            
fig

### Embedding Classification

#### Get the data

In [6]:
def get_sat_embedding(aoi, start_year, end_year):
     """
    Retrieves the satellite embedding data
    for the specified time range and area of interest (AOI).
    Returns:
        ee.Image: A median-composited, clipped image.
    """
     dataset = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

     image = dataset\
        .filterDate(f'{start_year}-01-01', f'{end_year}-01-01') \
         .filterBounds(aoi)
     mosaicked = image.mosaic().clip(aoi)
     return mosaicked
#get the satellite embedding data
data_2020 = get_sat_embedding(aoi, 2017, 2018)
#sample = geemap.shp_to_ee('C:/Data Spasial/Adaptive workflow testing/GPS-points/GT_2016/GT_2016_epistem_OganIlir.shp')

#### Classification

In [7]:
#extract the pixel from embedding data
from luma_ge.classification import FeatureExtraction, Generate_LULC
features = FeatureExtraction()
clf = Generate_LULC()
train_embed, test_embed = features.stratified_split(sample, data_2020, 
                            class_prop='ID_epistem', train_ratio=0.75, seed=42)
#applied hard classification/original MS
classification_embed, model_embed = clf.hard_classification(training_data = train_embed, #Ms only training data
 class_property='ID_epistem', 
 image=data_2020, 
 ntrees=350,
 v_split=15,
 return_model=True)

Stratified Random Split Training Pixel Size: 547
Stratified Random Split Testing Pixel Size: 153


In [ ]:
#Evaluate model performance
print("Evaluating embedding classification.")
try:
    model_acc_embed = clf.evaluate_model(
        trained_model=model_embed,
        test_data=test_embed,
        class_property='ID_epistem'
    )
    
except Exception as e:
    print(f"❌ Error in model evaluation: {e}")                                                    
import pandas as pd
# Display accuracy results
print("=== Model Performance Summary ===")
print(f"Overall Accuracy: {model_acc_embed['overall_accuracy']:.4f} ({model_acc_embed['overall_accuracy']*100:.2f}%)")
print(f"Kappa Coefficient: {model_acc_embed['kappa']:.4f}")
print(f"Overall G-Mean: {model_acc_embed['overall_gmean']:.4f}")
print("\n=== Per-Class Metrics ===")
#Class Dataframe
metrics_embed = pd.DataFrame({
    'Precision': model_acc_embed['precision'],
    'Recall': model_acc_embed['recall'],
    'F1-Score': model_acc_embed['f1_scores'],
    'G-Mean': model_acc_embed['gmean_per_class']
})
#Round to 4 decimal places
metrics_embed = metrics_embed.round(4)
display(metrics_embed)

### Visualization

In [ ]:
class_ids = list(ogan_ilir_class.keys())
vis_params = {
    "min": min(class_ids),
    "max": max(class_ids),
    "palette": [ogan_ilir_class[i]["color"] for i in class_ids]
}

legend_dict = {
    ogan_ilir_class[i]["name"]: ogan_ilir_class[i]["color"]
    for i in class_ids
}
m = geemap.Map()
m.centerObject(aoi, 8)
m.addLayer(classification_raw, vis_params, "MS_Only_Classification")
m.addLayer(classification_embed, vis_params, "embedding classification")
m.add_legend(title="Land Cover", legend_dict=legend_dict)
m


In [ ]:
export_task = ee.batch.Export.image.toDrive(
    image=classification_raw,
    description='Classification_ogan_MS',
    folder='Earth Engine',
    fileNamePrefix='Classification_ogan_MS',
    scale=30,
    region=aoi.geometry(),  # or aoi.geometry()
    maxPixels=1e13
)
export_task.start()
import time

while export_task.active():
    print('Exporting... (status: {})'.format(export_task.status()['state']))
    time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

In [ ]:
export_task = ee.batch.Export.image.toDrive(
    image=classification_embed,
    description='Classification_ogan_Embedding_rev',
    folder='Earth Engine',
    fileNamePrefix='Classification_ogan_Embedding_rev',
    scale=10,
    region=aoi.geometry(),  # or aoi.geometry()
    maxPixels=1e13
)
export_task.start()
import time

while export_task.active():
    print('Exporting... (status: {})'.format(export_task.status()['state']))
    time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

## Soft Classification

### Soft Classification using Landsat data

In [8]:
soft_ms = clf.soft_classification(training_data=train, class_property='ID_epistem',
                                  image=predictor_stack, 
                                  include_final_map=True,
                                  ntrees=350,
                                  v_split=9)

Creating final classification using argmax


In [9]:
class_ids = list(ogan_ilir_class.keys())
vis_params = {
    "min": min(class_ids),
    "max": max(class_ids),
    "palette": [ogan_ilir_class[i]["color"] for i in class_ids]
}

legend_dict = {
    ogan_ilir_class[i]["name"]: ogan_ilir_class[i]["color"]
    for i in class_ids
}
classification = soft_ms.select('classification')
m = geemap.Map()
m.centerObject(aoi, 8)
m.addLayer(classification, vis_params, "MS_Only_Classification")
m.add_legend(title="Land Cover", legend_dict=legend_dict)
m


NameError: name 'ogan_ilir_class' is not defined

In [ ]:
export_task = ee.batch.Export.image.toDrive(
    image=classification,
    description='Classification_ogan_Landsat_soft',
    folder='Earth Engine',
    fileNamePrefix='Classification_ogan_Landsat_soft',
    scale=30,
    region=aoi.geometry(),  # or aoi.geometry()
    maxPixels=1e13
)
export_task.start()
import time

while export_task.active():
    print('Exporting... (status: {})'.format(export_task.status()['state']))
    time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

In [10]:
embed_ms = clf.soft_classification(training_data=train_embed, class_property='ID_epistem',
                                  image=data_2020, 
                                  include_final_map=True,
                                  ntrees=300,
                                  v_split=15)
soft_clf_embed = embed_ms.select('classification')
#m.addLayer(soft_clf_embed, vis_params, "Embedding soft classification")
#m

Creating final classification using argmax


In [11]:
export_task = ee.batch.Export.image.toDrive(
    image=soft_clf_embed,
    description='Classification_ogan_Embed_soft',
    folder='Earth Engine',
    fileNamePrefix='Classification_ogan_Embed_soft',
    scale=10,
    region=aoi.geometry(),  # or aoi.geometry()
    maxPixels=1e13
)
export_task.start()
import time

while export_task.active():
    print('Exporting... (status: {})'.format(export_task.status()['state']))
    time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Exporting... (status: READY)
Export complete (status: CANCELLED)
